# 01 — Ingest, validate, clean, enrich

Part 1 (26 marks). Run with **Restart Kernel → Run All**. Outputs are written to the **repo root** (hand-off files for Parts 2–3; `data/` keeps only the three originals): `rejected_rows.json`, `clinics_enriched.csv`, `appointments_clean.csv`, `load_data.sql`.

## Setup — importing & path setup

In [2]:
lazy import csv
lazy import json
lazy import requests
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().parent
data_folder = root / "data"
appointments_csv = data_folder / "appointments.csv"
clinics_csv = data_folder / "clinics.csv"

## Task 1.1 — Read the file the foundations way (csv module only)

### 1.1.1 Open with DictReader — header & row count

In [3]:
with open(appointments_csv, newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print("Header:", header)
print("Rows read:", len(raw_rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020


### 1.1.2 `parse_row(row)`

In [ ]:
int_fields = ("age", "fee_inr", "prior_no_shows")
float_fields = ("distance_km",)


def parse_row(row: dict) -> dict:
    """Return a typed copy of one CSV row; raise ValueError if a numeric field won't convert."""
    parsed = dict(row)
    for field in int_fields:
        try:
            parsed[field] = int(row[field])
        except ValueError:
            raise ValueError(f"Invalid integer for '{field}': {row[field]!r}") from None
    for field in float_fields:
        try:
            parsed[field] = float(row[field])
        except ValueError:
            raise ValueError(f"Invalid float for '{field}': {row[field]!r}") from None
    return parsed

### 1.1.3 Accept / reject loop

In [5]:
accepted, rejected = [], []
reasons = Counter()

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as err:
        rejected.append(row)
        reasons[str(err).split("'")[1]] += 1

print(f"Accepted: {len(accepted)}, Rejected: {len(rejected)}")
print("Rejected by field:", dict(reasons))

Accepted: 988, Rejected: 32
Rejected by field: {'distance_km': 20, 'age': 12}


### 1.1.4 Save `rejected_rows.json`

In [6]:
with open(root / "rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print(f"Saved {len(rejected)} rows to rejected_rows.json")

Saved 32 rows to rejected_rows.json


A type check is not a validity check: rows with impossible ages (e.g. 250) are *accepted* here because they are valid integers. Validity is handled in Task 1.3.

## Task 1.2 — Inspect and clean, pass 1 (pandas, fresh from raw CSV)

### 1.2.0 Load & inspect — shape, info, nulls, duplicates

In [14]:
df = pd.read_csv(appointments_csv)

print("Shape:", df.shape)
df.info()
print("\nNulls per column:\n", df.isnull().sum(), sep="")
print("\nExact duplicate rows:", df.duplicated().sum())
print("booking_channel raw spellings:", df["booking_channel"].nunique())

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB

Nulls per column:
appointment_id      0
patient_id          0
clinic_id           0
book

### 1.2(a) Drop exact duplicates

In [15]:
df = df.drop_duplicates()
print("Rows after (a) drop duplicates:", len(df))

Rows after (a) drop duplicates: 1000


### 1.2(b) Clean `booking_channel`

In [16]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after (b):", len(df))
print("booking_channel distinct after cleaning:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))

Rows after (b): 1000
booking_channel distinct after cleaning: 3 ['App', 'Phone', 'Walk-In']


### 1.2(c) Convert `appt_date` & `booked_on`

In [17]:
had_appt_date = df["appt_date"].notna()
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

unreadable = (had_appt_date & df["appt_date"].isna()).sum()
print("Rows after (c):", len(df))
print("Unparseable appt_date text -> NaT:", unreadable, "(plus 1 'N/A' pandas already read as missing)")
print("Missing appt_date total:", df["appt_date"].isna().sum(), "| missing status:", df["status"].isna().sum())

Rows after (c): 1000
Unparseable appt_date text -> NaT: 5 (plus 1 'N/A' pandas already read as missing)
Missing appt_date total: 6 | missing status: 4


### 1.2(d) Drop rows missing `appt_date` / `status`

In [18]:
df = df.dropna(subset=["appt_date", "status"]).reset_index(drop=True)
print("Rows after (d):", len(df))

Rows after (d): 990


## Task 1.3 — Clean, pass 2: validity, gaps, outliers

### 1.3(a) Invalid ages → missing

In [19]:
age_min, age_max = 1, 100

invalid_age = (df["age"] < age_min) | (df["age"] > age_max)
print("Invalid ages:", invalid_age.sum(), "->", sorted(df.loc[invalid_age, "age"].tolist()))
df.loc[invalid_age, "age"] = np.nan
print("Missing ages now:", df["age"].isna().sum())

Invalid ages: 5 -> [-4.0, 0.0, 150.0, 199.0, 250.0]
Missing ages now: 17


### 1.3(b) Fill `age` & `distance_km` with medians

In [20]:
age_median = df["age"].median()
distance_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Median age:", age_median, "| median distance_km:", distance_median)
print("Missing after fill -> age:", df["age"].isna().sum(), "distance_km:", df["distance_km"].isna().sum())

Median age: 48.0 | median distance_km: 8.25
Missing after fill -> age: 0 distance_km: 0


### 1.3(c) IQR caps on `distance_km` & `fee_inr`

In [21]:
iqr_multiplier = 1.5

for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - iqr_multiplier * iqr, q3 + iqr_multiplier * iqr
    n_low, n_high = (df[col] < low).sum(), (df[col] > high).sum()
    df[col] = np.clip(df[col], low, high)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, fences=({low}, {high}) -> capped {n_low + n_high} (low {n_low}, high {n_high})")

distance_km: Q1=4.625, Q3=13.0, IQR=8.375, fences=(-7.9375, 25.5625) -> capped 26 (low 0, high 26)
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, fences=(68.75, 1218.75) -> capped 3 (low 0, high 3)


### 1.3(d) Round

In [22]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Largest fee after capping:", df["fee_inr"].max())
print("distance_km range:", df["distance_km"].min(), "-", df["distance_km"].max())

Largest fee after capping: 1219
distance_km range: 0.6 - 25.6


## Task 1.4 — Features and a NumPy summary

### 1.4.0 `lead_days` & `no_show`

In [23]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "-", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean() * 100:.1f}%)")

lead_days range: 1 - 45
No-shows: 218 of 990 (22.0%)


### 1.4.1 `X` array

In [24]:
features = ["age", "distance_km", "fee_inr", "lead_days"]
X = df[features].to_numpy()
print("X shape:", X.shape)

X shape: (990, 4)


### 1.4(a) Column means

In [25]:
col_means = X.mean(axis=0)
print(dict(zip(features, col_means.round(2).tolist())))

{'age': 48.17, 'distance_km': 9.5, 'fee_inr': 635.71, 'lead_days': 11.82}


### 1.4(b) `lead_days > 14` mask

In [26]:
lead_days = X[:, features.index("lead_days")]
mask = lead_days > 14
print(f"lead_days > 14: {mask.sum()} rows ({mask.mean() * 100:.1f}%)")

lead_days > 14: 296 rows (29.9%)


### 1.4(c) z-scores of `distance_km`

In [27]:
x = X[:, features.index("distance_km")]
z = (x - np.mean(x)) / np.std(x)          # np.std = population std (ddof=0)
print("Rows with |z| > 2:", (np.abs(z) > 2).sum())

Rows with |z| > 2: 53


## Task 1.5 — Enrich the clinics from a live API

### 1.5.1 Load clinics

In [28]:
clinics = pd.read_csv(clinics_csv)
clinics

,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462


### 1.5.2 Open-Meteo call per clinic (timeout, retry once, None on failure)

In [29]:
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
TIMEOUT_S = 10
ATTEMPTS = 2   # first try + one retry


def fetch_temp_c(lat: float, lon: float) -> tuple[float | None, str | None]:
    """Return (temp_c, None) on success or (None, reason) on failure. Never invents a value."""
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    reason = None
    for _ in range(ATTEMPTS):
        try:
            response = requests.get(OPEN_METEO_URL, params=params, timeout=TIMEOUT_S)
        except requests.exceptions.RequestException as err:
            reason = f"request failed: {err}"
            continue                                    # retry once
        if response.status_code != 200:
            return None, f"status {response.status_code}"
        data = response.json()
        if "hourly" not in data:
            return None, "reply has no 'hourly' field"
        return data["hourly"]["temperature_2m"][0], None
    return None, reason


temps = []
for clinic in clinics.itertuples():
    temp_c, reason = fetch_temp_c(clinic.latitude, clinic.longitude)
    if reason:
        print(f"{clinic.clinic_name}: temp_c = None ({reason})")
    temps.append(temp_c)

clinics["temp_c"] = temps
clinics

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.1
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.5
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,18.9
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,18.9
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.3
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,19.3
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,19.0


### 1.5.3 Save `clinics_enriched.csv`

In [30]:
clinics.to_csv(root / "clinics_enriched.csv", index=False)
print(f"Saved clinics_enriched.csv: {clinics.shape}, missing temp_c: {clinics['temp_c'].isna().sum()}")

Saved clinics_enriched.csv: (8, 7), missing temp_c: 0


## Task 1.6 — Export the clean table and write the SQL load script

### 1.6.1 Checks — unique `appointment_id`, no missing values

In [31]:
EXPORT_COLUMNS = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
    "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show",
]

ids_unique = df["appointment_id"].is_unique
missing_total = df[EXPORT_COLUMNS].isnull().sum().sum()
print(f"appointment_id unique: {ids_unique} ({df['appointment_id'].nunique()} distinct)")
print("Missing values across the 16 columns:", missing_total)

assert ids_unique, "Duplicate appointment_id values"
assert missing_total == 0, "Missing values in export columns"

appointment_id unique: True (990 distinct)
Missing values across the 16 columns: 0


### 1.6.2 Sort, format dates, save `appointments_clean.csv` (16 columns)

In [32]:
clean = df.sort_values("appointment_id")[EXPORT_COLUMNS].reset_index(drop=True)
for col in ["booked_on", "appt_date"]:
    clean[col] = clean[col].dt.strftime("%Y-%m-%d")

clean.to_csv(root / "appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)

Saved appointments_clean.csv: (990, 16)


### 1.6.3 Write `load_data.sql` — clinics first, then appointments

In [34]:
def sql_value(value) -> str:
    """Text in single quotes, numbers unquoted, missing as NULL."""
    if pd.isna(value):
        return "NULL"
    if isinstance(value, str):
        return f"'{value}'"
    return f"{value}"


def insert_statement(table: str, row) -> str:
    return f"INSERT INTO {table} VALUES ({','.join(sql_value(v) for v in row)});"


lines = []
for row in clinics.itertuples(index=False):          # clinics first: appointments.clinic_id is a foreign key
    lines.append(insert_statement("clinics", row))
for row in clean.itertuples(index=False):
    lines.append(insert_statement("appointments", row))

with open(root / "data warehouse" / "load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")

n_clinics = sum(l.startswith("INSERT INTO clinics") for l in lines)
print(f"load_data.sql: {n_clinics} + {len(lines) - n_clinics} = {len(lines)} INSERT statements")
print(lines[0])
print(lines[n_clinics])

load_data.sql: 8 + 990 = 998 INSERT statements
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.1);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
